In [0]:
import requests
import json
import os
from datetime import datetime
import time


dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)

# keys load 
alpha_key = dbutils.secrets.get(
    scope="fin-project",
    key="alpha_vantage_key"
)


if not alpha_key:
    raise ValueError("❌ Key not found.")

print(f"✅ Alpha Vantage key loaded: {alpha_key[:4]}****")
print(f"✅ Run timestamp: {datetime.now()}")

In [0]:
TICKERS = [
    "RELIANCE.BSE",
    "TCS.BSE",
    "INFY.BSE",
    "HDFCBANK.BSE",
    "ICICIBANK.BSE",
    "HINDUNILVR.BSE",
    "SBIN.BSE",
    "BAJFINANCE.BSE",
    "BHARTIARTL.BSE",
    "KOTAKBANK.BSE"
]

BASE_URL = "https://www.alphavantage.co/query"
RUN_DATE = datetime.now().strftime("%Y-%m-%d")

print(f"✅ Tracking {len(TICKERS)} tickers")
print(f"✅ Run date: {RUN_DATE}")

In [0]:
def fetch_single_stock(ticker: str) -> dict:
    """
    Fetch daily OHLCV data for one stock ticker.
    Returns raw JSON response from Alpha Vantage.
    """
    params = {
        "function"   : "TIME_SERIES_DAILY",
        "symbol"     : ticker,
        "outputsize" : "compact",
        "apikey"     : alpha_key
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if "Time Series (Daily)" not in data:
            print(f"⚠️ WARNING: No data for {ticker}")
            print(f"   Response: {data}")
            return None

        rows = len(data["Time Series (Daily)"])
        print(f"✅ {ticker} — {rows} days fetched")
        return data

    except requests.exceptions.Timeout:
        print(f"❌ TIMEOUT: {ticker}")
        return None

    except requests.exceptions.RequestException as e:
        print(f"❌ ERROR: {ticker} — {e}")
        return None

In [0]:
def fetch_all_stocks() -> list:
    """
    Loop through all tickers.
    Sleep 15s between calls — free tier limit.
    """
    all_data = []
    total    = len(TICKERS)
    failed   = []

    print(f"Starting ingestion for {total} tickers...\n")

    for i, ticker in enumerate(TICKERS):
        print(f"[{i+1}/{total}] Fetching {ticker}...")
        data = fetch_single_stock(ticker)

        if data:
            record = {
                "ticker"     : ticker,
                "fetched_at" : datetime.now().isoformat(),
                "run_date"   : RUN_DATE,
                "source"     : "alpha_vantage",
                "data"       : data
            }
            all_data.append(record)
        else:
            failed.append(ticker)

        if i < total - 1:
            print(f"   Waiting 15 seconds...\n")
            time.sleep(10)

    print(f"\n{'='*40}")
    print(f"✅ Success : {len(all_data)}/{total} tickers")
    if failed:
        print(f"❌ Failed  : {failed}")
    print(f"{'='*40}")

    return all_data


all_stock_data = fetch_all_stocks()

In [0]:
def validate_stock_data(all_data: list) -> bool:
    """
    Basic quality checks before Bronze layer.
    """
    print("Running validation checks...\n")

    if len(all_data) == 0:
        print("❌ VALIDATION FAILED: No data fetched")
        return False

    passed = 0
    failed = 0

    for record in all_data:
        ticker      = record["ticker"]
        time_series = record["data"].get("Time Series (Daily)", {})
        row_count   = len(time_series)
        latest_date = max(time_series.keys()) if time_series else "N/A"

        if row_count >= 10:
            print(f"✅ {ticker} — {row_count} rows — latest: {latest_date}")
            passed += 1
        else:
            print(f"⚠️ {ticker} — only {row_count} rows — check API")
            failed += 1

    print(f"\nValidation: {passed} passed, {failed} warnings")
    return True

# validate
is_valid = validate_stock_data(all_stock_data)
print(f"\nResult: {'✅ PASSED' if is_valid else '❌ FAILED'}")

In [0]:
# verify Bronze table
df_verify = spark.sql("""
    SELECT 
        ticker,
        COUNT(*)    as total_rows,
        MIN(date)   as earliest_date,
        MAX(date)   as latest_date,
        MIN(close)  as min_close,
        MAX(close)  as max_close
    FROM financial_market.bronze.stocks
    GROUP BY ticker
    ORDER BY ticker
""")

print("✅ Bronze table — financial_market.bronze.stocks")
print(f"✅ Total tickers: {df_verify.count()}")
print()
display(df_verify)

In [0]:
def save_to_bronze(all_data: list):
    """
    Save raw stock data to Bronze Delta table.
    Partitioned by ticker + run_date.
    """
    if not all_data:
        print("❌ No data to save")
        return

    
    rows = []
    for record in all_data:
        ticker      = record["ticker"]
        fetched_at  = record["fetched_at"]
        run_date    = record["run_date"]
        time_series = record["data"].get("Time Series (Daily)", {})

        for date, values in time_series.items():
            rows.append({
                "ticker"     : ticker,
                "date"       : date,
                "open"       : float(values.get("1. open", 0)),
                "high"       : float(values.get("2. high", 0)),
                "low"        : float(values.get("3. low", 0)),
                "close"      : float(values.get("4. close", 0)),
                "volume"     : float(values.get("5. volume", 0)),
                "fetched_at" : fetched_at,
                "run_date"   : run_date,
                "source"     : "alpha_vantage"
            })

    
    df = spark.createDataFrame(rows)

    print(f"✅ Total rows to save: {df.count()}")
    print("\nSchema:")
    df.printSchema()

    # saved in Bronze Delta table 
    df.write \
        .format("delta") \
        .mode("Overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("ticker", "run_date") \
        .saveAsTable("financial_market.bronze.stocks")

    print(f"\n✅ Saved to: financial_market.bronze.stocks")
    print(f"✅ Partitioned by: ticker + run_date")


save_to_bronze(all_stock_data)

In [0]:
# quick verify
df = spark.sql("""
    SELECT ticker, COUNT(*) as rows
    FROM financial_market.bronze.stocks
    GROUP BY ticker
    ORDER BY ticker
""")
print(f"Total rows: {spark.table('financial_market.bronze.stocks').count()}")
display(df)